# 11장 Fine-Tuning Representation Models for Classification — SageMaker AI Studio(JupyterLab)용

<!-- origin-notice -->
> **원본 노트북:** [chapter11/Chapter 11 - Fine-Tuning BERT.ipynb](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter11/Chapter%2011%20-%20Fine-Tuning%20BERT.ipynb)  
> **원본 저장소:** [HandsOnLLM/Hands-On-Large-Language-Models](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models) (Apache License 2.0, © 2024 Jay Alammar & Maarten Grootendorst)  
> 이 노트북은 원본을 SageMaker AI Studio에서 실행되도록 수정하고 한국어 설명·보충 실험을 추가한 것입니다. 변경 내용은 노트북 안의 "책과 다를 수 있는 점"에 적혀 있습니다.

원본: HandsOnLLM/Hands-On-Large-Language-Models `chapter11` 노트북(Apache 2.0)을 현재 라이브러리 버전에 맞게 수정. 4장에서는 사전학습 모델을 **얼린(frozen)** 채 분류에 썼습니다. 이 장은 BERT 계열 표현 모델의 가중치를 직접 **미세조정(fine-tuning)**합니다.

| 절 | 하는 일 | 책 위치(PDF 쪽) |
|---|---|---|
| 1 | BERT 전체를 영화 리뷰 감성 분류로 미세조정 | 446–451 |
| 2 | 층 얼리기(freezing): 분류 헤드만 / 앞 10개 블록 얼리고 비교 | 452–458 |
| 3 | SetFit: 클래스당 16개 예시만으로 퓨샷(few-shot) 분류 | 459–467 |
| 4 | 마스크 언어 모델링(MLM)으로 사전학습 이어 하기(continued pretraining) | 468–474 |
| 5 | 개체명 인식(NER): 문서가 아니라 **토큰마다** 분류 | 475–485 |

**권장 환경:** `ml.g4dn.xlarge`(T4 16GB), 스토리지 30GB 이상. 전체 실행 약 40~60분(대부분 4절 MLM 10에폭). `(+ 보충)`은 책에 없는 실험입니다.

**빠른 실행:** 아래 `QUICK = True`로 바꾸면 데이터 일부와 에폭 1회로 흐름만 10분 안팎에 확인합니다. 점수는 책보다 낮게 나옵니다.

## 0. 환경 점검과 패키지 설치

이 장은 라이브러리 API 변화의 영향을 많이 받습니다(`Trainer(tokenizer=)` 제거, `datasets.load_metric` 제거, 데이터셋 스크립트 지원 종료 등). 그래서 2장처럼 "없는 것만 설치"하지 않고, **이 노트북을 검증한 버전 조합으로 맞춥니다**(2026-11 기준 transformers 5.20.2 / datasets 5.2.0 / setfit 1.2.0). 설치가 일어나면 **커널을 재시작**한 뒤 이 셀부터 다시 실행하세요.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h ~ | tail -1
import importlib.metadata as md, subprocess, sys
PINS = {"transformers": "5.20.2", "datasets": "5.2.0", "accelerate": "1.15.0", "evaluate": "0.4.6",
        "seqeval": "1.2.2", "setfit": "1.2.0", "sentence-transformers": "6.2.0"}
def ver(p):
    try: return md.version(p)
    except md.PackageNotFoundError: return None
todo = [f"{p}=={v}" for p, v in PINS.items() if ver(p) != v]
if todo:
    print("설치:", todo)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *todo])
    print("\n설치 완료 → 메뉴 Kernel > Restart Kernel 후 이 셀부터 다시 실행하세요.")
else:
    print("검증된 버전 그대로입니다.")

In [ ]:
import gc, torch, numpy as np, pandas as pd, transformers
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "| transformers", transformers.__version__)

QUICK = False          # True: 데이터 일부·에폭 축소로 흐름만 확인
N_TRAIN, N_TEST = (1000, 300) if QUICK else (None, None)
results = {}           # 절마다 F1을 모아 마지막에 비교

def free_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

## 1. 사전학습 BERT 미세조정 (PDF 446–451)

4장의 "작업 특화 모델"을 이번에는 직접 만듭니다. 사전학습된 `bert-base-cased` 위에 분류 헤드(feedforward 층 하나)를 얹고, **BERT 본체와 헤드를 함께** 학습합니다(그림 11-2, 11-3). 역전파가 헤드에서 시작해 BERT 전체로 흘러간다는 점이 4장과의 차이입니다.

데이터는 4장과 같은 Rotten Tomatoes 영화 리뷰(긍정 5,331 / 부정 5,331)입니다.

In [ ]:
from datasets import load_dataset

# 책은 "rotten_tomatoes"(짧은 이름). 최신 datasets/huggingface_hub에서는 "소유자/이름" 형식이 필요
tomatoes = load_dataset("cornell-movie-review-data/rotten_tomatoes")
train_data, test_data = tomatoes["train"], tomatoes["test"]
if QUICK:
    train_data = train_data.shuffle(seed=42).select(range(N_TRAIN))
    test_data = test_data.shuffle(seed=42).select(range(N_TEST))
print(train_data, test_data)

`num_labels=2`를 주면 그 크기에 맞는 분류 헤드가 새로 만들어집니다. 헤드는 무작위로 초기화되므로 "일부 가중치가 새로 초기화되었다"는 로드 리포트가 나오는 것이 정상입니다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_id = "bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)

토큰화합니다. `DataCollatorWithPadding`은 배치마다 **그 배치의 가장 긴 문장 길이**에 맞춰 패딩합니다. 전체 데이터를 최대 길이로 맞추는 것보다 계산이 적습니다.

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_test = test_data.map(preprocess_function, batched=True)

평가 지표는 F1입니다.

> **책과 다른 점.** 책의 `from datasets import load_metric`은 datasets 3.0에서 제거되었습니다. 같은 기능을 하는 `evaluate.load("f1")`로 바꿨습니다.

In [ ]:
import evaluate
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"f1": f1_metric.compute(predictions=predictions, references=labels)["f1"]}

> **책과 다른 점.** ① `Trainer(tokenizer=...)`는 transformers 5에서 `processing_class=`로 바뀌었습니다. ② `save_total_limit=1`을 추가했습니다. 에폭마다 체크포인트(BERT base 기준 옵티마이저 상태 포함 약 1.3GB)를 남기므로, 4절처럼 10에폭을 돌리면 디스크가 10GB 넘게 찹니다.

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    "model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    save_strategy="epoch",
    save_total_limit=1,          # 추가: 체크포인트 1개만 유지
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,  # 책: tokenizer=tokenizer
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

학습 후 평가합니다. T4에서 1에폭 약 3~5분. **관찰 포인트:** 책은 F1 0.85입니다. 4장에서 미세조정 없이 쓴 작업 특화 모델은 0.80이었습니다.

In [ ]:
trainer.train()
r = trainer.evaluate(); print(r)
results["1. 전체 미세조정"] = r["eval_f1"]

## 2. 층 얼리기 (PDF 452–458)

계산을 줄이려고 일부 층을 얼리면(`requires_grad=False`) 성능이 얼마나 떨어지는지 봅니다. 먼저 모델을 새로 불러와 층 이름을 확인합니다. 임베딩 → 인코더 블록 0~11 → pooler → classifier 순서입니다(그림 11-4).

In [ ]:
del model, trainer; free_gpu()
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)

names = [name for name, _ in model.named_parameters()]
print(len(names), "개 파라미터 텐서")
print("\n".join(names[:6] + ["..."] + names[-6:]))

**분류 헤드만 학습:** BERT 본체를 전부 얼립니다(그림 11-5). 4장의 "임베딩 + 분류기"와 비슷한 상태입니다.

In [ ]:
for name, param in model.named_parameters():
    param.requires_grad = name.startswith("classifier")

def count_trainable(m):
    t = sum(p.numel() for p in m.parameters() if p.requires_grad)
    a = sum(p.numel() for p in m.parameters())
    return t, a
t, a = count_trainable(model); print(f"학습 파라미터 {t:,} / 전체 {a:,} ({t/a:.4%})")

trainer = Trainer(model=model, args=training_args, train_dataset=tokenized_train, eval_dataset=tokenized_test,
                  processing_class=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics)
trainer.train()
r = trainer.evaluate(); print(r)
results["2a. 헤드만 학습"] = r["eval_f1"]

**관찰 포인트:** 학습이 눈에 띄게 빨라지지만 F1은 책 기준 0.63으로 크게 떨어집니다.

다음은 **앞쪽 10개 인코더 블록을 얼리는** 경우입니다(그림 11-6). 책 코드는 파라미터 순번 165 미만을 얼립니다.

> **책 주석 읽는 법.** 책 주석은 "Encoder block 11 starts at index 165"입니다. 블록 번호를 1부터 셀 때의 11번째, 즉 이름으로는 `layer.10`입니다. 아래 셀이 순번 165의 이름을 출력해 확인합니다. 결과적으로 `layer.10`, `layer.11`, pooler, classifier가 학습됩니다.

In [ ]:
del model, trainer; free_gpu()
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)

all_names = [n for n, _ in model.named_parameters()]
print("index 164:", all_names[164]); print("index 165:", all_names[165])

for index, (name, param) in enumerate(model.named_parameters()):
    if index < 165:
        param.requires_grad = False
t, a = count_trainable(model); print(f"학습 파라미터 {t:,} / 전체 {a:,} ({t/a:.2%})")

trainer = Trainer(model=model, args=training_args, train_dataset=tokenized_train, eval_dataset=tokenized_test,
                  processing_class=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics)
trainer.train()
r = trainer.evaluate(); print(r)
results["2b. 블록 0~9 얼림"] = r["eval_f1"]

**(+ 보충) 세 설정 비교.** 학습 파라미터 비율과 F1을 나란히 봅니다. 책 그림 11-7은 "앞 5개 블록만 학습해도 전체 학습에 거의 근접한다"는 결과를 보여 줍니다. 전부 다 학습하지 않아도 된다는 점이 12장의 LoRA(일부만 학습)로 이어집니다.

In [ ]:
pd.Series(results, name='F1').round(3).to_frame()

In [ ]:
del model, trainer; free_gpu()

## 3. SetFit: 적은 예시로 분류하기 (PDF 459–467)

레이블 데이터가 클래스당 몇 개뿐일 때 쓰는 방법입니다. SetFit은 세 단계로 동작합니다(그림 11-13).

1. **문장 쌍 생성:** 같은 클래스끼리는 긍정 쌍, 다른 클래스끼리는 부정 쌍으로 만듭니다.
2. **임베딩 모델 미세조정:** 그 쌍으로 sentence-transformers 모델을 대조 학습(contrastive learning, 10장)합니다.
3. **분류기 학습:** 미세조정된 임베딩 위에서 로지스틱 회귀를 학습합니다.

전체 8,530개 대신 **클래스당 16개, 총 32개**만 씁니다.

In [ ]:
from setfit import sample_dataset
sampled_train_data = sample_dataset(tomatoes["train"], num_samples=16)
sampled_train_data

In [ ]:
from setfit import SetFitModel
model = SetFitModel.from_pretrained("sentence-transformers/all-mpnet-base-v2")

`num_iterations=20`은 예시 하나당 만들 문장 쌍 수입니다. 32개 × 20 × (긍정·부정) 2 = **1,280쌍**이 됩니다. 출력의 `Num unique pairs`로 확인하세요.

> **책 코드의 `args.eval_strategy = args.evaluation_strategy`** 줄은 SetFit과 transformers 사이 인자 이름이 바뀌던 시기의 호환용입니다. setfit 1.2.0에서는 두 속성이 모두 있어 그대로 두어도 동작합니다.

In [ ]:
from setfit import TrainingArguments as SetFitTrainingArguments
from setfit import Trainer as SetFitTrainer

args = SetFitTrainingArguments(
    num_epochs=3,          # 대조 학습 에폭
    num_iterations=20,     # 예시당 생성할 문장 쌍 수
)
args.eval_strategy = args.evaluation_strategy

trainer = SetFitTrainer(
    model=model,
    args=args,
    train_dataset=sampled_train_data,
    eval_dataset=test_data,
    metric="f1",
)
trainer.train()

**관찰 포인트:** 32개만으로 책 기준 F1 약 0.84입니다. 1절에서 8,530개로 전체 미세조정한 0.85와 거의 같습니다.

In [ ]:
r = trainer.evaluate(); print(r)
results["3. SetFit (32개)"] = r["f1"]

책의 TIP처럼 로지스틱 회귀 대신 학습 가능한 분류 헤드를 쓰려면 `use_differentiable_head=True, head_params={"out_features": 클래스 수}`로 불러옵니다. 결과만 확인하고 넘어갑니다.

In [ ]:
head_model = SetFitModel.from_pretrained("sentence-transformers/all-mpnet-base-v2",
                                         use_differentiable_head=True, head_params={"out_features": 2})
print(type(head_model.model_head).__name__)
del model, trainer, head_model; free_gpu()

## 4. MLM으로 사전학습 이어 하기 (PDF 468–474)

일반 텍스트(위키백과·책)로 사전학습된 BERT는 특정 분야의 말투를 잘 모릅니다. 미세조정 전에 **우리 도메인 텍스트로 마스크 언어 모델링(MLM)을 더 학습**하면 단어 표현이 그 분야 쪽으로 옮겨 갑니다(그림 11-15, 11-16). 일반 BERT → BioBERT → 약품 분류 모델로 가는 흐름과 같습니다. 레이블은 쓰지 않으므로 지웁니다.

In [ ]:
from transformers import AutoModelForMaskedLM
model = AutoModelForMaskedLM.from_pretrained("bert-base-cased")
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_train = train_data.map(preprocess_function, batched=True).remove_columns("label")
tokenized_test = test_data.map(preprocess_function, batched=True).remove_columns("label")

토큰의 15%를 무작위로 가리는 collator입니다.

> **책과 다른 점.** 책이 단어 전체 가리기(whole-word masking)용으로 소개한 `DataCollatorForWholeWordMask`는 transformers 5에서 없어졌습니다. 대신 `DataCollatorForLanguageModeling(..., whole_word_mask=True)`를 씁니다.

In [ ]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)

책 코드는 10에폭입니다(본문에는 "20 epochs"라고 적혀 있지만 코드 값은 10). T4에서 약 25~35분 걸립니다. `QUICK = True`면 1에폭만 돌립니다.

In [ ]:
mlm_args = TrainingArguments(
    "model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1 if QUICK else 10,
    weight_decay=0.01,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
)
trainer = Trainer(model=model, args=mlm_args, train_dataset=tokenized_train, eval_dataset=tokenized_test,
                  processing_class=tokenizer, data_collator=data_collator)

tokenizer.save_pretrained("mlm")   # 토크나이저는 학습 중 바뀌지 않음
trainer.train()
model.save_pretrained("mlm")

학습 전후로 `"What a horrible [MASK]!"`의 빈칸을 채워 봅니다. **관찰 포인트:** 원래 모델은 idea·dream·day 같은 일반 단어를, 이어 학습한 모델은 movie·film·comedy 같은 영화 리뷰 단어를 고르는지 보세요.

> **책과 다른 점.** 책의 `f">>> {pred["sequence"]}"`처럼 f-string 안에 같은 종류의 따옴표를 겹쳐 쓰는 문법은 Python 3.12부터 허용됩니다. 이미지 버전과 관계없이 동작하도록 작은따옴표로 바꿨습니다.

In [ ]:
from transformers import pipeline
for name in ["bert-base-cased", "mlm"]:
    mask_filler = pipeline("fill-mask", model=name)
    print(f"--- {name}")
    for pred in mask_filler("What a horrible [MASK]!"):
        print(f">>> {pred['sequence']}")

**(+ 보충) 숫자로 확인하기.** 같은 테스트 리뷰에서 MLM 손실(가린 토큰 맞히기의 교차 엔트로피)을 비교합니다. `exp(loss)`는 퍼플렉서티(perplexity)로, 12장 "단어 수준 지표"에 나오는 그 값입니다. 낮을수록 그 분야 문장을 덜 "당황하며" 예측한다는 뜻입니다. 가리는 위치가 무작위라 실행마다 조금씩 달라집니다.

In [ ]:
import math
ppl = {}
for name in ["bert-base-cased", "mlm"]:
    torch.manual_seed(0)
    m = AutoModelForMaskedLM.from_pretrained(name)
    ev = Trainer(model=m, args=mlm_args, eval_dataset=tokenized_test,
                 processing_class=tokenizer, data_collator=data_collator).evaluate()
    ppl[name] = {"eval_loss": round(ev["eval_loss"], 3), "perplexity": round(math.exp(ev["eval_loss"]), 1)}
    del m
pd.DataFrame(ppl).T

이어 학습한 모델을 분류에 쓰려면 1절과 같은 방식으로 `mlm` 폴더에서 불러오면 됩니다(책 474쪽). 여기서는 불러오기만 확인합니다.

In [ ]:
cls_from_mlm = AutoModelForSequenceClassification.from_pretrained("mlm", num_labels=2)
print(type(cls_from_mlm).__name__)
del model, trainer, cls_from_mlm; free_gpu()

## 5. 개체명 인식 NER (PDF 475–485)

문서 하나에 레이블 하나가 아니라 **토큰마다** 레이블을 붙입니다(그림 11-19). 사람(PER)·조직(ORG)·장소(LOC)·기타(MISC)를 찾으며, 개인정보 비식별화에 많이 쓰입니다. 데이터는 CoNLL-2003 영어판(학습 약 14,000문장)입니다.

> **책과 다른 점.** 책의 `load_dataset("conll2003", trust_remote_code=True)`는 데이터셋 **로딩 스크립트**를 실행하는 방식인데, datasets 4.0부터 스크립트 지원이 끝났습니다. Hugging Face가 같은 데이터를 parquet로 변환해 둔 `refs/convert/parquet` 리비전을 읽습니다. 848번 예시까지 책과 같습니다.

In [ ]:
dataset = load_dataset("eriktks/conll2003", revision="refs/convert/parquet")
if QUICK:
    dataset["train"] = dataset["train"].select(range(3000))
    dataset["test"] = dataset["test"].select(range(N_TEST))
example = dataset["train"][848]
example

레이블 앞의 `B-`는 개체의 시작(Beginning), `I-`는 이어지는 부분(Inside)입니다. "Dean Palmer"는 `B-PER I-PER`라서 한 사람으로 묶입니다(그림 11-20).

In [ ]:
label2id = {"O": 0, "B-PER": 1, "I-PER": 2, "B-ORG": 3, "I-ORG": 4,
            "B-LOC": 5, "I-LOC": 6, "B-MISC": 7, "I-MISC": 8}
id2label = {index: label for label, index in label2id.items()}
print(dataset["train"].features["ner_tags"].feature.names)   # 데이터셋 정의와 순서가 같은지 확인

In [ ]:
from transformers import AutoModelForTokenClassification
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased", num_labels=len(id2label), id2label=id2label, label2id=label2id)

데이터는 **단어** 단위로 레이블이 붙어 있는데, 토크나이저는 `homer`를 `home` + `##r`로 쪼갭니다. 레이블 개수가 맞지 않게 됩니다.

In [ ]:
token_ids = tokenizer(example["tokens"], is_split_into_words=True)["input_ids"]
tokenizer.convert_ids_to_tokens(token_ids)

그래서 레이블을 토큰에 맞춰 다시 붙입니다(그림 11-21). 단어의 첫 토큰은 원래 레이블, 이어지는 토큰은 `B-`를 `I-`로 바꿉니다(레이블 번호가 홀수면 +1). `[CLS]`·`[SEP]`는 `-100`으로 두어 손실 계산에서 빠지게 합니다.

In [ ]:
def align_labels(examples):
    token_ids = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = examples["ner_tags"]
    updated_labels = []
    for index, label in enumerate(labels):
        word_ids = token_ids.word_ids(batch_index=index)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx != previous_word_idx:          # 새 단어의 시작
                previous_word_idx = word_idx
                label_ids.append(-100 if word_idx is None else label[word_idx])
            elif word_idx is None:                     # 특수 토큰
                label_ids.append(-100)
            else:                                      # 같은 단어의 이어지는 토큰: B-XXX → I-XXX
                updated_label = label[word_idx]
                if updated_label % 2 == 1:
                    updated_label += 1
                label_ids.append(updated_label)
        updated_labels.append(label_ids)
    token_ids["labels"] = updated_labels
    return token_ids

tokenized = dataset.map(align_labels, batched=True)
print(f"Original: {example['ner_tags']}")
print(f"Updated: {tokenized['train'][848]['labels']}")

평가는 `seqeval`로 합니다. 책 코드는 토큰 하나하나를 길이 1짜리 시퀀스로 넘깁니다. seqeval은 원래 문장 단위 레이블 시퀀스를 받아 **개체 단위**로 채점하므로, 이 방식은 사실상 토큰 단위에 가까운 점수가 됩니다. 책 결과와 맞추려고 그대로 둡니다.

In [ ]:
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=2)
    true_predictions, true_labels = [], []
    for prediction, label in zip(predictions, labels):
        for token_prediction, token_label in zip(prediction, label):
            if token_label != -100:
                true_predictions.append([id2label[token_prediction]])
                true_labels.append([id2label[token_label]])
    results_ = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {"f1": results_["overall_f1"]}

In [ ]:
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=tokenized["train"], eval_dataset=tokenized["test"],
    processing_class=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics,
)
trainer.train()
r = trainer.evaluate(); print(r)
results["5. NER (토큰 F1)"] = r["eval_f1"]

저장한 모델로 추론합니다. **관찰 포인트:** "Maarten"이 `Ma` `##arte` `##n` 세 토큰으로 나뉘고 각각 `B-PER`, `I-PER`, `I-PER`가 붙습니다.

In [ ]:
trainer.save_model("ner_model")
token_classifier = pipeline("token-classification", model="ner_model")
token_classifier("My name is Maarten.")

**(+ 보충) 토큰을 개체로 묶기.** 실제로는 조각난 토큰보다 "Maarten = 사람" 같은 **개체 단위** 결과가 필요합니다. `aggregation_strategy="simple"`이 B/I 태그를 따라 토큰을 합칩니다. 주행 로그나 사고 보고서에서 회사·지명을 뽑는 상황을 가정한 문장으로도 시험해 봅니다.

In [ ]:
ner = pipeline("token-classification", model="ner_model", aggregation_strategy="simple")
for s in ["My name is Maarten.",
          "Waymo tested its robotaxi in Phoenix after talks with Toyota in Tokyo."]:
    print(s)
    for e in ner(s):
        print(f"   {e['entity_group']:5s} {e['word']:12s} {e['score']:.3f}")

## 6. 정리

In [ ]:
pd.Series(results, name='F1').round(3).to_frame()

| 방법 | 학습하는 것 | 필요한 레이블 | 이 노트북 |
|---|---|---|---|
| 전체 미세조정 | BERT 전체 + 분류 헤드 | 수천 개 | 1절 |
| 층 얼리기 | 뒤쪽 블록 + 헤드(또는 헤드만) | 수천 개 | 2절 |
| SetFit | 임베딩 모델(대조 학습) + 분류기 | 클래스당 수십 개 | 3절 |
| 사전학습 이어 하기(MLM) | BERT 전체, 레이블 없이 | 없음(도메인 텍스트) | 4절 |
| 토큰 분류(NER) | BERT 전체 + 토큰별 헤드 | 단어 단위 태그 | 5절 |

**자율주행·Agent로 이어지는 점(책 밖).** 주행 데이터 파이프라인에서는 "어떤 장면인가"를 분류하는 작업이 많습니다(장면 태깅, 사고 보고서 분류, 로그의 이상 유형 분류). 레이블이 적은 신규 시나리오에는 3절 SetFit 방식이, 사내 용어가 많은 문서에는 4절의 도메인 사전학습이 출발점이 됩니다. Agent 쪽에서는 5절 NER과 1절 분류기가 LLM 앞단의 **경량 라우터·필터**(질의 유형 분류, 개인정보 마스킹)로 자주 쓰입니다. 큰 생성 모델을 매번 부르는 것보다 빠르고 저렴하기 때문입니다.

**끝나면** Studio에서 space를 **Stop** 하세요. `model/`, `mlm/`, `ner_model/` 폴더(합계 약 2~3GB)는 EBS에 남습니다. 필요 없으면 지워 두세요.